In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
import timm

In [2]:
#ViT
vision_encoder = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=True
)

#BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_encoder = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def get_image_feature(images):
    features = vision_encoder.forward_features(images)
    image_cls = features[:,0]
    return image_cls

def get_text_feature(texts):
    inputs = tokenizer(
        texts,
        padding = True,
        truncation = True,
        return_tensors="pt"
    )

    outputs = text_encoder(**inputs)
    text_cls = outputs.last_hidden_state[:,0]

    return text_cls

images = torch.randn(4, 3, 224, 224)

texts = [
    "a photo of a dog",
    "a photo of a cat",
    "a photo of a car",
    "a photo of a bird"
]

image_cls = get_image_feature(images)
text_cls = get_text_feature(texts)

image_projection = nn.Linear(192,256)
text_projection = nn.Linear(768,256)

image_embedding = image_projection(image_cls)
text_embedding = text_projection(text_cls)

In [4]:
image_embedding = F.normalize(image_embedding, dim=-1)
text_embedding = F.normalize(text_embedding, dim=-1)

logits = image_embedding @ text_embedding.T

In [5]:
print(image_embedding.shape)
print(text_embedding.shape)
print(logits.shape)

torch.Size([4, 256])
torch.Size([4, 256])
torch.Size([4, 4])


In [6]:
labels = torch.arange(logits.size(0))

#image -> Text
loss_i = F.cross_entropy(logits, labels)
loss_t = F.cross_entropy(logits.T,labels)

loss = (loss_i + loss_t) / 2
print(loss)

tensor(1.3863, grad_fn=<DivBackward0>)
